# Buscar un usuario por id

## Qué hace

Muestra el usuario Ada, buscado por su id.

## Qué pasa en la base de datos

Solo lee. No cambia filas.

## Qué acción usa

`GET /users/:id` con el id de Ada. El id se busca antes en `GET /users`, por el correo `ada@example.com`.

## Anotaciones

Si Ada no está en la lista, no hay id y se ve `Id is invalid`. Si no hay usuario con ese id, se ve `User not found`. Si no se puede leer, se ve `Could not find the user`.

Se abre con `pnpm jupyter-node-http` desde la app.

In [ ]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "src" / "logic").is_dir():
    app = app.parent
if not (app / "src" / "logic").is_dir():
    raise SystemExit("No encuentro la app.")

code = r"""
import Table from "cli-table3";

const origin = "http://127.0.0.1:3004";
const usersResponse = await fetch(origin + "/users");
const users = await usersResponse.json();
let adaId;
for (const user of users) {
  if (user.email === "ada@example.com") {
    adaId = user.id;
  }
}
const response = await fetch(origin + "/users/" + adaId);
const result = await response.json();
if (response.ok === false) {
  console.log(result.message);
} else {
  const table = new Table({
    head: ["id", "name", "email", "createdAt", "updatedAt"],
    style: { head: ["cyan", "bold"] },
  });
  table.push([
    result.id,
    result.name,
    result.email,
    result.createdAt,
    result.updatedAt,
  ]);
  console.log(table.toString());
}
"""

env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")